# Session 3: Exercises

**Do these before session 4.** They take about ninety minutes; the ⭐⭐⭐ ones join the session's ideas
together.

Same pattern as last week: **predict, then measure.** Write down what you expect before running a cell.
The gap between your prediction and the result shows you what you actually believed. Predictions and
answers go in the cells marked **✍️**; code to write is marked `# TODO`.

Worked solutions are released after the next session.

## Setup (just run it)

The client with retries, the inbox, `as_block`, `ask_schema`, and the v0.2 schemas.

In [1]:
import json
import re
import sys
from collections import Counter
from pathlib import Path
from typing import Literal

import yaml
from google import genai
from google.genai import types
from pydantic import BaseModel, Field, ValidationError, ValidationInfo, field_validator

sys.path.insert(0, str(Path.cwd() / "project"))
from config import settings                                          # noqa: E402
from schemas import BANGLA_DIGITS, ORDER_NUMBER, TicketFacts, TriageResult   # noqa: E402
from triage import triage_system                                     # noqa: E402

client = genai.Client(http_options=types.HttpOptions(
    retry_options=types.HttpRetryOptions(attempts=4, http_status_codes=[429, 500, 503])))
MODEL = settings.MODEL
RAW_TICKETS = yaml.safe_load((settings.DATA_DIR / "tickets.yml").read_text(encoding="utf-8"))
by_id = {t["id"]: t for t in RAW_TICKETS}


def as_block(t: dict) -> str:
    return f"<ticket>\nSubject: {t['subject']}\n{t['body']}\n</ticket>"


def ask_schema(schema, contents, system=None):
    '''One structured call; returns the whole response (lesson section 2).'''
    config = {"response_mime_type": "application/json", "response_schema": schema}
    if system:
        config["system_instruction"] = system
    return client.models.generate_content(model=MODEL, contents=contents, config=config)


print(f"{MODEL} · {len(RAW_TICKETS)} tickets")

gemini-3.5-flash-lite · 20 tickets


## ⭐ Exercise 1: Read the error before you run it

**The idea.** Lesson section 1.3. A `ValidationError` is a list of structured problems, and reading
one quickly is a skill you'll use every day. Here's a ticket with several things wrong.

**✍️ Predict first:** how many errors will Pydantic report, and for each one, which field (`loc`) and
which `type`? Write them down before you run the cell.

**✍️ Your prediction:**

3 errors: `id` (pattern), `category` (literal) and `received` (date).

In [2]:
from schemas import Ticket

bad = {"id": "TCK-1", "subject": "help", "body": "", "from": "a@example.com",
       "category": "Shipping", "received": "yesterday"}

# TODO: validate `bad` with Ticket.model_validate and print loc, type and msg for every error.
try:
    Ticket.model_validate(bad)
except ValidationError as e:
    print(e.error_count(), "errors")
    for err in e.errors():
        print(err["loc"], "|", err["type"], "|", err["msg"])

4 errors
('id',) | string_pattern_mismatch | String should match pattern '^TCK-\d{3}$'
('body',) | string_too_short | String should have at least 1 character
('category',) | literal_error | Input should be 'billing', 'complaint', 'order_status', 'other', 'product_question', 'refund', 'return', 'shipping' or 'warranty'
('received',) | date_from_datetime_parsing | Input should be a valid date or datetime, input is too short


**✍️ Your answer.** How many did you predict? Which one surprised you?

I predicted 3 and got 4. I missed `body`: an empty string fails `min_length=1`.

## ⭐ Exercise 2: The confidence field

**The idea.** It's tempting to ask the model how sure it is. Add a `confidence` field to triage and
run it on an ambiguous ticket and an obvious one, three times each.

**✍️ Predict first:** will the ambiguous ticket (`TCK-003`, an angry "where is my order") get lower
confidence than the obvious one (`TCK-016`, "please cancel ORD-5002")?

**✍️ Your prediction:**

Yes, TCK-003 should get medium or low at least once.

In [3]:
# TODO: define WithConfidence: TriageResult's category field, plus
#       confidence: Literal["low", "medium", "high"], declared after the category.
# Then run it three times on each ticket and print category and confidence.
from schemas import TicketCategory


class WithConfidence(BaseModel):
    category: TicketCategory = Field(description="What the ticket is about. Pick the single best fit.")
    confidence: Literal["low", "medium", "high"] = Field(
        description="How sure you are about the category.")


system = triage_system(with_examples=False)
for tid in ["TCK-003", "TCK-016"]:
    for i in range(3):
        r = ask_schema(WithConfidence, as_block(by_id[tid]), system)
        print(tid, i + 1, r.parsed.category, r.parsed.confidence)

TCK-003 1 complaint high
TCK-003 2 complaint high
TCK-003 3 complaint high
TCK-016 1 order_status high
TCK-016 2 order_status high
TCK-016 3 order_status high


**✍️ Your answer.** Did confidence track how hard the ticket is? Would you ship this field?

No, both got "high" all 6 times, so it doesn't track difficulty. I wouldn't ship it.

## ⭐⭐ Exercise 3: Tracking numbers

**The idea.** Lesson sections 3.3 and 3.4, on a new field. Courier tracking numbers look like
`SW-88231` (TCK-017 has one). Add `tracking_numbers` to a facts model with the same pair of validators:
normalise first, then check against the ticket. Then score it over the inbox: found, missed, invented.

**✍️ Predict first:** how many tracking numbers are in the inbox, and will the model invent any?

**✍️ Your prediction:**

Just 1 (TCK-017's SW-88231), and none invented thanks to the validator.

In [4]:
TRACKING = re.compile(r"^(SW)?[\s\-#]*(\d+)$", re.IGNORECASE)


class Tracking(BaseModel):
    tracking_numbers: list[str] = Field(description="Every courier tracking number (SW-NNNNN) exactly as the customer wrote it. "
                                                    "Not order numbers (ORD-...) or refund references. "
                                                    "Empty list if there is none. Never invent one.")    # TODO: describe it, like TicketFacts.order_ids

    # TODO: a mode="before" validator that turns "sw 88231" or "SW-৮৮২৩১" into "SW-88231"
    #       and rejects anything that isn't shaped like a tracking number.
    @field_validator("tracking_numbers", mode="before")
    @classmethod
    def normalise(cls, nums: list) -> list[str]:
        clean = []
        for raw in nums:
            text = str(raw).strip().translate(BANGLA_DIGITS)
            if not text:
                continue
            m = TRACKING.match(text)
            if m is None:
                raise ValueError(f"{raw!r} is not a tracking number")
            clean.append(f"SW-{m.group(2)}")
        return clean

    # TODO: an after validator that raises if a number isn't in the ticket text from the context.
    #       Return early when info.context is None: the SDK runs your validators too, without
    #       context, when it fills .parsed (lesson section 3.4).
    @field_validator("tracking_numbers")
    @classmethod
    def in_ticket(cls, nums: list[str], info: ValidationInfo) -> list[str]:
        if info.context is None:
            return nums
        text = info.context["ticket"].translate(BANGLA_DIGITS)
        for n in nums:
            if n.removeprefix("SW-") not in text:
                raise ValueError(f"{n} is not in the ticket")
        return nums


def score_tracking():
    found = missed = invented = 0
    for t in RAW_TICKETS:
        text = f"{t['subject']}\n{t['body']}"
        truth = {f"SW-{n}" for n in re.findall(r"SW-(\d+)", text.translate(BANGLA_DIGITS))}
        r = ask_schema(Tracking, as_block(t))
        try:
            got = set(Tracking.model_validate_json(r.text, context={"ticket": text}).tracking_numbers)
        except ValidationError:
            got = set()
        found += len(got & truth)
        missed += len(truth - got)
        invented += len(got - truth)
    print(f"found {found} · missed {missed} · invented {invented}")


score_tracking()

found 1 · missed 0 · invented 0


**✍️ Your answer.** What did the score show, and what would the score have hidden if you'd averaged missed and invented into one accuracy number?

Found 1, missed 0, invented 0. One accuracy number would look near 100% because of the 19 empty tickets, and it would hide that inventing a number is much worse than missing one.

## ⭐⭐ Exercise 4: The limit that usually works

**The idea.** Lesson section 2, step 4. Measure how often `.parsed` comes back `None` for three caps
on `summary` (40, 60 and 80 characters), each with a description that **agrees** with the cap and one
that **disagrees** ("under 20 words").

**✍️ Predict first:** which of the six combinations fails most? Does any reach zero?

**✍️ Your prediction:**

Cap 40 with "under 20 words" will fail most. None will reach zero.

In [7]:
from pydantic import create_model


def none_rate(cap: int, description: str, tickets=RAW_TICKETS[:10]) -> int:
    '''How many of `tickets` come back with .parsed None for this cap and description.'''
    Schema = create_model("Schema", summary=(str, Field(max_length=cap, description=description)))
    # TODO: run ask_schema(Schema, as_block(t)) for each ticket and count .parsed is None
    count = 0
    for t in tickets:
        r = ask_schema(Schema, as_block(t))
        if r.parsed is None:
            count += 1
    return count


# TODO: for cap in (40, 60, 80), print none_rate with an agreeing description
#       (e.g. f"One sentence, at most {cap} characters.") and with "One sentence, under 20 words."
for cap in (40, 60, 80):
    agree = none_rate(cap, f"One sentence, at most {cap} characters.")
    disagree = none_rate(cap, "One sentence, under 20 words.")
    print(f"cap {cap}: agreeing {agree}/10 None · disagreeing {disagree}/10 None")

cap 40: agreeing 1/10 None · disagreeing 10/10 None
cap 60: agreeing 0/10 None · disagreeing 8/10 None


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://aistudio.google.com/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 15, model: gemini-3.5-flash-lite, description: Generate content requests per minute limit exceeded on Free Tier. See https://aistudio.google.com/docs/rate-limits.\nPlease retry in 39.776240622s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://aistudio.google.com/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'description': 'Generate content requests per minute limit exceeded on Free Tier. See https://aist

**✍️ Your answer.** What made failures rarer, and what would make them *handled*?

A description that agrees with the cap made failures rarer (cap 40: 10→1, cap 60: 7→0). To handle them: check `.parsed is None` and retry, or enforce the length in a validator.

## ⭐⭐ Exercise 5: Earn your examples

**The idea.** Lesson section 4.1. The shipped examples bought nothing. Write three new ones aimed at the
tickets the rules still miss (the `complaint` vs `order_status` and `return` vs `warranty` borderlines),
then compare rules-only with rules-plus-your-examples, **twice each**.

Write your examples fresh. Don't copy inbox tickets: an example in the prompt is never a test case.

**✍️ Predict first:** will targeted examples beat the rules by more than the noise (about ±2 tickets)?

**✍️ Your prediction:**

Maybe +1 or 2, but within the ±2 noise.

In [12]:
print("C223101")

def score(system: str) -> int:
    hits = 0
    for t in RAW_TICKETS:
        r = ask_schema(TriageResult, as_block(t), system)
        if r.parsed is not None and r.parsed.category == t["category"]:
            hits += 1
    return hits


MY_EXAMPLES = '''
Examples of how ShopWise labels the awkward cases:

<ticket>
Subject: package late, getting annoyed
It's been 8 days and my speaker still hasn't come. Can someone tell me where it is? This is really annoying.
</ticket>
category: order_status

<ticket>
Subject: smartwatch stopped charging
My watch was delivered 4 months ago and now it won't charge at all. Is this covered? I'd like it fixed.
</ticket>
category: warranty

<ticket>
Subject: don't want the earbuds anymore
Got the earbuds last week, never opened the box. I changed my mind and want to send them back.
</ticket>
category: return
'''   # TODO: three <ticket>...</ticket> + category: ... examples, written fresh

rules = triage_system(with_examples=False)
# TODO: print score(rules) twice and score(rules + MY_EXAMPLES) twice
print("rules only:      ", score(rules), score(rules), "/", len(RAW_TICKETS))
print("rules + examples:", score(rules + MY_EXAMPLES), score(rules + MY_EXAMPLES), "/", len(RAW_TICKETS))

C223101
rules only:       18 18 / 20
rules + examples: 18 17 / 20


**✍️ Your answer.** Was the difference bigger than the run-to-run spread? What would you ship?

No: 18, 18 vs 18, 17 is within the noise. I'd ship rules only, since it's shorter and cheaper.

## ⭐⭐⭐ Exercise 6: Code decides the queue

**The idea.** "The model advises, code decides", with no model call at all. The lesson wrote the CTO's
spreadsheet to `triage.csv`. Give urgency a service-level target with an `Enum` (lesson appendix A),
then produce the queue the support team works from: soonest SLA first, and within the same SLA,
tickets inside the return window before unknown ones before those outside.

**✍️ Predict first:** which ticket ends up at the top, and why?

**✍️ Your prediction:**

TCK-003, the angry high-urgency ticket.

In [14]:
print("C223101")
import csv
from enum import Enum

with open("triage.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))


class Urgency(str, Enum):
    LOW = "low"
    MEDIUM = "medium"
    HIGH = "high"
    # TODO: an sla_hours property: high 4, medium 24, low 72
    @property
    def sla_hours(self) -> int:
        return {"high": 4, "medium": 24, "low": 72}[self.value]


class QueueRow(BaseModel):
    id: str
    urgency: Urgency
    needs_human: bool
    return_window: Literal["inside", "unknown", "outside"]
    summary: str


# TODO: validate every row into a QueueRow (note: needs_human arrives as the string "True"),
#       sort by (sla_hours, return-window order), and print the queue.
WINDOW_ORDER = {"inside": 0, "unknown": 1, "outside": 2}

queue = [QueueRow.model_validate(r) for r in rows]   # pydantic turns "True"/"False" into a bool
queue.sort(key=lambda q: (q.urgency.sla_hours, WINDOW_ORDER[q.return_window]))

for q in queue:
    human = "human" if q.needs_human else ""
    print(f"{q.id}  {q.urgency.sla_hours:>2}h  {q.return_window:8} {human:6} {q.summary}")

C223101
TCK-002   4h  inside   human  Customer wants a refund because the left earcup crackles.
TCK-003   4h  unknown  human  Customer demands shipment or refund for delayed kettle order after no replies.
TCK-009   4h  unknown  human  Customer charged twice for a single order and requests a refund.
TCK-011   4h  unknown  human  Customer received a backpack instead of the ordered smartwatch.
TCK-019   4h  unknown  human  Customer complains about ignored emails and broken chat, threatening to leave.
TCK-015  24h  inside   human  Customer asking if rapid overnight battery drain is a product fault.
TCK-001  24h  unknown         Customer is asking for the tracking status and delivery date of order ORD-5001.
TCK-004  24h  unknown         Customer is asking for the delivery status of order ORD-5003.
TCK-012  24h  unknown         Customer wants to change the delivery address for order ORD-5002.
TCK-014  24h  unknown  human  Customer asking why gold membership did not include free shipping.
TCK

**✍️ Your answer.** Which parts of that queue came from the model, and which from code? Where would you change the SLA if the manager asked?

Urgency, needs_human and summary came from the model. The SLA hours, return window and sort order are code, which is why TCK-002 (inside the window) went first. To change the SLA I'd edit only `sla_hours`.

## ⭐⭐⭐ Exercise 7: When does caching start?

**The idea.** Lesson section 5. Find the smallest prefix that gets an implicit cache hit on your key,
by growing a library of policy documents and asking the same question three times at each size. Then
estimate how large v0.2's system slot would have to grow before caching mattered.

**✍️ Predict first:** at roughly how many tokens does the first hit appear?

**✍️ Your prediction:**

Around 1,000 tokens, so even 4 docs should hit.

In [16]:
policy = Path("../session-02/returns_policy.md").read_text(encoding="utf-8")


def library(n_docs: int) -> str:
    return "\n".join(f"<document index='{i}'>\n{policy.replace('ShopWise', f'Store{i}')}\n</document>"
                     for i in range(n_docs))


def cached_on_repeat(system: str) -> int | None:
    '''Ask the same question three times; return what the third call reports as cached.'''
    # TODO: three generate_content calls with config={"system_instruction": system};
    #       return the last call's usage_metadata.cached_content_token_count
    for _ in range(3):
        r = client.models.generate_content(
            model=MODEL, contents="How many days do I have to return an item at Store0?",
            config={"system_instruction": system})
    return r.usage_metadata.cached_content_token_count


# TODO: for n in (4, 8, 12, 16, 20, 24): print the token count of library(n) and cached_on_repeat(library(n))
for n in (4, 8, 12, 16, 20, 24):
    lib = library(n)
    tokens = client.models.count_tokens(model=MODEL, contents=lib).total_tokens
    print(f"{n:2} docs · {tokens:5} tokens · cached: {cached_on_repeat(lib)}")

 4 docs ·  1612 tokens · cached: None
 8 docs ·  3224 tokens · cached: None
12 docs ·  4840 tokens · cached: 2032
16 docs ·  6460 tokens · cached: 4073
20 docs ·  8080 tokens · cached: 4077
24 docs ·  9700 tokens · cached: 6121


**✍️ Your answer.** Where did caching start, and what does that mean for v0.2 and for session 2's date-first decision?

The first hit came at 12 docs (~4,800 tokens), with none at 3,200. v0.2's prompt (~170–700 tokens) is far below that, so putting the date first costs nothing yet; move it to the end if the prompt grows past ~4k.